In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
print(GEMINI_API_KEY)

NameError: name 'GEMINI_API_KEY' is not defined

In [51]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

In [28]:
loader = PyPDFLoader("../data/Aditya_Goenka_Resume_Latest.pdf")
doc = loader.load()


In [29]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=100)
chunks = text_splitter.split_documents(doc)
embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vec_embedding = embedding.embed_documents([chunk.page_content for chunk in chunks])
len(vec_embedding)



4

In [30]:


vector_store = Chroma(
    collection_name="my_collection",
    embedding_function=embedding,
    persist_directory="../data/chroma_db"
)

In [31]:
vector_store.add_documents(chunks)

['ad093a36-2fb5-4bca-a1e2-a56f78c3b335',
 '704a201e-63fa-403e-a623-ea40adf4139c',
 'c33a207c-9099-4b83-8ff0-eaf6103a1314',
 'af2faeb1-83d6-494a-b031-17eca58c63f7']

In [48]:
llm = GoogleGenerativeAI(model="gemini-3-flash-preview")

In [49]:
def get_context(query:str):
    results = vector_store.similarity_search(query)
    context = "\n\n".join([f"Context {i+1}: {result.page_content}" for i, result in enumerate(results)])
    return {"context": context, "query": query}

In [52]:
prompt=PromptTemplate.from_template("""
    You are a helpful assistant. Answer the question based on the context below. If the answer is not contained within the text below, say "I don't know".
    context: {context}
    Question: {query}
""")

In [53]:
rag_chain = get_context | prompt | llm

In [54]:
rag_chain.invoke("what is the name of the person in the resume?")

'The name of the person in the resume is Aditya Goenka.'